In [2]:
"""
LSTM model for coral bleaching severity prediction.

Input:  52-week sequences of 7 thermal stress features
Output: 4-class ordinal bleaching severity (none/low/moderate/severe)

Architecture:
  - Bidirectional LSTM (captures both buildup and recent cooling)
  - 2 LSTM layers with dropout
  - Attention mechanism (learns which weeks matter most)
  - Static features (lat, lon) concatenated before classification head
  - Ordinal-aware loss option
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import time
import os

In [3]:
# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
SEED = 42
BATCH_SIZE = 128
EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15  # Early stopping patience
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_LEN = 52
N_FEATURES = 7
N_CLASSES = 4
HIDDEN_SIZE = 128
N_LAYERS = 2
DROPOUT = 0.3

np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
# ──────────────────────────────────────────────────────────────
# DATASET
# ──────────────────────────────────────────────────────────────
class BleachingDataset(Dataset):
    def __init__(self, X, y, meta):
        # Normalize features per-feature across the dataset
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        
        # Static features: lat, lon (normalized)
        self.static = torch.FloatTensor(meta[:, :2])  # lat, lon
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.static[idx], self.y[idx]

In [5]:
# ──────────────────────────────────────────────────────────────
# ATTENTION LAYER
# ──────────────────────────────────────────────────────────────
class TemporalAttention(nn.Module):
    """Learns which timesteps are most important for prediction."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1, bias=False),
        )
    
    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_size)
        scores = self.attn(lstm_output).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(scores, dim=1)            # (batch, seq_len)
        context = torch.bmm(
            weights.unsqueeze(1), lstm_output
        ).squeeze(1)  # (batch, hidden_size)
        return context, weights

In [6]:
# ──────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────
class BleachingLSTM(nn.Module):
    def __init__(
        self,
        n_features=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        n_layers=N_LAYERS,
        n_classes=N_CLASSES,
        n_static=2,
        dropout=DROPOUT,
    ):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=hidden_size // 2,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        
        # Attention over LSTM outputs
        self.attention = TemporalAttention(hidden_size * 2)  # *2 for bidirectional
        
        # Classification head
        # Combines: attention context + last hidden state + static features
        head_input_size = hidden_size * 2 + hidden_size * 2 + n_static
        
        self.classifier = nn.Sequential(
            nn.Linear(head_input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_size // 2, n_classes),
        )
    
    def forward(self, x_seq, x_static):
        # x_seq: (batch, 52, 7)
        # x_static: (batch, 2)
        
        # Project input features
        x = self.input_proj(x_seq)  # (batch, 52, hidden//2)
        
        # LSTM
        lstm_out, (h_n, _) = self.lstm(x)  # lstm_out: (batch, 52, hidden*2)
        
        # Attention-weighted context
        attn_context, attn_weights = self.attention(lstm_out)  # (batch, hidden*2)
        
        # Last timestep hidden state (concat forward and backward)
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden*2)
        
        # Combine everything
        combined = torch.cat([attn_context, last_hidden, x_static], dim=1)
        
        logits = self.classifier(combined)  # (batch, n_classes)
        return logits, attn_weights

In [7]:
# ──────────────────────────────────────────────────────────────
# TRAINING UTILITIES
# ──────────────────────────────────────────────────────────────
def get_class_weights(y):
    """Inverse frequency weighting for imbalanced classes."""
    counts = Counter(y.tolist())
    total = sum(counts.values())
    weights = {c: total / (len(counts) * n) for c, n in counts.items()}
    return torch.FloatTensor([weights[i] for i in range(N_CLASSES)]).to(DEVICE)


def get_weighted_sampler(y):
    """WeightedRandomSampler for balanced batches."""
    counts = Counter(y.tolist())
    class_weights = {c: 1.0 / n for c, n in counts.items()}
    sample_weights = [class_weights[label] for label in y.tolist()]
    return WeightedRandomSampler(sample_weights, len(sample_weights))


def normalize_features(X_train, X_val, X_test):
    """Per-feature normalization using training set statistics."""
    # X shape: (N, 52, 7)
    mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
    std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)
    std[std == 0] = 1  # Avoid division by zero
    
    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std
    
    return X_train_norm, X_val_norm, X_test_norm, mean, std


def normalize_static(meta_train, meta_val, meta_test):
    """Normalize lat/lon."""
    mean = meta_train[:, :2].mean(axis=0)
    std = meta_train[:, :2].std(axis=0)
    std[std == 0] = 1
    
    meta_train_norm = meta_train.copy()
    meta_val_norm = meta_val.copy()
    meta_test_norm = meta_test.copy()
    
    meta_train_norm[:, :2] = (meta_train[:, :2] - mean) / std
    meta_val_norm[:, :2] = (meta_val[:, :2] - mean) / std
    meta_test_norm[:, :2] = (meta_test[:, :2] - mean) / std
    
    return meta_train_norm, meta_val_norm, meta_test_norm

In [9]:
# ──────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ──────────────────────────────────────────────────────────────
def train():
    # Load data
    print("=" * 70)
    print("Loading data")
    print("=" * 70)
    data = np.load("datasets/sequences.npz")
    X, y, meta = data["X"], data["y"], data["meta"]
    print(f"  X: {X.shape}, y: {y.shape}, meta: {meta.shape}")
    print(f"  Device: {DEVICE}")
    
    # Stratified split: 70/15/15
    X_trainval, X_test, y_trainval, y_test, m_trainval, m_test = train_test_split(
        X, y, meta, test_size=0.15, stratify=y, random_state=SEED
    )
    X_train, X_val, y_train, y_val, m_train, m_val = train_test_split(
        X_trainval, y_trainval, m_trainval, test_size=0.176, stratify=y_trainval, random_state=SEED
    )  # 0.176 of 0.85 ≈ 0.15 of total
    
    print(f"  Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")
    print(f"  Train class dist: {dict(Counter(y_train.tolist()))}")
    
    # Normalize
    X_train, X_val, X_test, feat_mean, feat_std = normalize_features(X_train, X_val, X_test)
    m_train, m_val, m_test = normalize_static(m_train, m_val, m_test)
    
    # Datasets and loaders
    train_ds = BleachingDataset(X_train, y_train, m_train)
    val_ds = BleachingDataset(X_val, y_val, m_val)
    test_ds = BleachingDataset(X_test, y_test, m_test)
    
    sampler = get_weighted_sampler(y_train)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Model
    model = BleachingLSTM().to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  Model parameters: {total_params:,} ({trainable_params:,} trainable)")
    
    # Loss and optimizer
    class_weights = get_class_weights(y_train)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Training
    print("\n" + "=" * 70)
    print("Training")
    print("=" * 70)
    
    best_val_loss = float("inf")
    best_val_acc = 0
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    
    for epoch in range(EPOCHS):
        t0 = time.time()
        
        # Train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for x_seq, x_static, labels in train_loader:
            x_seq = x_seq.to(DEVICE)
            x_static = x_static.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(x_seq, x_static)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
            train_correct += (logits.argmax(1) == labels).sum().item()
            train_total += labels.size(0)
        
        scheduler.step()
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validate
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for x_seq, x_static, labels in val_loader:
                x_seq = x_seq.to(DEVICE)
                x_static = x_static.to(DEVICE)
                labels = labels.to(DEVICE)
                
                logits, _ = model(x_seq, x_static)
                loss = criterion(logits, labels)
                
                val_loss += loss.item() * labels.size(0)
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        elapsed = time.time() - t0
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
            marker = " *"
        else:
            patience_counter += 1
            marker = ""
        
        if (epoch + 1) % 5 == 0 or epoch == 0 or marker:
            print(
                f"  Epoch {epoch+1:3d}/{EPOCHS} | "
                f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
                f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                f"{elapsed:.1f}s{marker}"
            )
        
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break
    
    # ──────────────────────────────────────────────────────────
    # EVALUATION
    # ──────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("Evaluation on test set")
    print("=" * 70)
    
    model.load_state_dict(torch.load("best_model.pt", weights_only=True))
    model.eval()
    
    all_preds = []
    all_labels = []
    all_attn = []
    
    with torch.no_grad():
        for x_seq, x_static, labels in test_loader:
            x_seq = x_seq.to(DEVICE)
            x_static = x_static.to(DEVICE)
            
            logits, attn_weights = model(x_seq, x_static)
            preds = logits.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_attn.append(attn_weights.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_attn = np.concatenate(all_attn, axis=0)
    
    class_names = ["None (0%)", "Low (1-10%)", "Moderate (10-50%)", "Severe (>50%)"]
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    print("Confusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"{'':>15} | " + " | ".join(f"{n:>8}" for n in class_names))
    print("-" * 70)
    for i, row in enumerate(cm):
        print(f"{class_names[i]:>15} | " + " | ".join(f"{v:>8d}" for v in row))
    
    # Attention analysis: which weeks matter most?
    print("\nAttention Analysis (mean attention weight by week):")
    mean_attn = all_attn.mean(axis=0)
    top_weeks = np.argsort(mean_attn)[::-1][:10]
    print(f"  Top 10 most attended weeks (0=oldest, 51=most recent):")
    for w in top_weeks:
        print(f"    Week {w:2d} (t-{52-w:2d} weeks before event): {mean_attn[w]:.4f}")
    
    # Save everything
    np.savez(
        "results.npz",
        predictions=all_preds,
        labels=all_labels,
        attention_weights=all_attn,
        history_train_loss=history["train_loss"],
        history_val_loss=history["val_loss"],
        history_val_acc=history["val_acc"],
        feat_mean=feat_mean,
        feat_std=feat_std,
    )
    print(f"\n  Results saved to results.npz")
    print(f"  Best model saved to best_model.pt")


In [10]:
if __name__ == "__main__":
    train()

Loading data
  X: (28539, 52, 7), y: (28539,), meta: (28539, 4)
  Device: cpu
  Train: 19988, Val: 4270, Test: 4281
  Train class dist: {1: 4495, 0: 11628, 2: 2497, 3: 1368}

  Model parameters: 701,892 (701,892 trainable)

Training
  Epoch   1/80 | Train Loss: 1.0495 Acc: 0.321 | Val Loss: 1.4439 Acc: 0.138 | LR: 1.00e-03 | 21.2s *
  Epoch   2/80 | Train Loss: 0.9753 Acc: 0.368 | Val Loss: 1.3288 Acc: 0.184 | LR: 9.98e-04 | 20.5s *
  Epoch   3/80 | Train Loss: 0.9343 Acc: 0.394 | Val Loss: 1.3142 Acc: 0.156 | LR: 9.97e-04 | 21.3s *
  Epoch   4/80 | Train Loss: 0.8852 Acc: 0.420 | Val Loss: 1.2384 Acc: 0.301 | LR: 9.94e-04 | 21.2s *
  Epoch   5/80 | Train Loss: 0.8451 Acc: 0.457 | Val Loss: 1.1718 Acc: 0.320 | LR: 9.90e-04 | 20.7s *
  Epoch   9/80 | Train Loss: 0.6960 Acc: 0.541 | Val Loss: 1.1067 Acc: 0.389 | LR: 9.69e-04 | 22.2s *
  Epoch  10/80 | Train Loss: 0.6525 Acc: 0.571 | Val Loss: 1.1532 Acc: 0.402 | LR: 9.62e-04 | 23.9s
  Epoch  15/80 | Train Loss: 0.5077 Acc: 0.656 | Val Lo